# 03 · SwiGLU feed-forward layers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/architecture/03_swiglu.ipynb) &nbsp; 📖 Concept deck: [SwiGLU](https://nilesh-salpe.github.io/visual-ai/architecture/04-swiglu/) &nbsp; ⏱️ Suggested time: **30 min**

**You will build:** the SwiGLU FFN used by Llama, PaLM and Mistral, the hidden-size rule that keeps its parameter count equal to a 4d ReLU FFN, and the exact sizes of real models.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def _silu(x):
    return x / (1 + np.exp(-x))

def _swiglu_ref(x, W_gate, W_up, W_down):
    return (_silu(x @ W_gate) * (x @ W_up)) @ W_down

## Task 1: The SwiGLU FFN

The 2017 FFN is $\max(0, xW_1)W_2$. Shazeer (2020) replaces it with a **gated** linear unit:
$$\mathrm{FFN}_{\mathrm{SwiGLU}}(x) = \big(\mathrm{Swish}(xW_{\text{gate}}) \odot xW_{\text{up}}\big)W_{\text{down}}, \qquad \mathrm{Swish}(z) = \mathrm{SiLU}(z) = z\,\sigma(z)$$
One projection decides *how much* of each hidden unit to let through (the gate), and the other carries the *value*. That makes three weight matrices instead of two. (Llama's code calls them `w1` = gate, `w3` = up and `w2` = down.)

In [ ]:
def silu(z):
    # TODO: z * sigmoid(z)
    raise NotImplementedError

def swiglu_ffn(x, W_gate, W_up, W_down):
    """x: [T, d]; W_gate, W_up: [d, h]; W_down: [h, d]. Returns [T, d]."""
    # TODO: (silu(x @ W_gate) * (x @ W_up)) @ W_down
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_swiglu_ffn(f):
    r = np.random.default_rng(1); d, h = 16, 40
    x = r.normal(size=(5, d)); Wg, Wu, Wd = r.normal(size=(d, h)), r.normal(size=(d, h)), r.normal(size=(h, d))
    y = f(x, Wg, Wu, Wd)
    assert y.shape == (5, d), f"shape {y.shape} != (5, {d})"
    silu_ = silu_solution if f.__name__.endswith('_solution') else silu
    assert np.allclose(silu_(np.array([0.0, 1.0, -1.0])), [0, 0.7310586, -0.2689414], atol=1e-6), "silu(z) must be z * sigmoid(z)"
    assert np.allclose(y, _swiglu_ref(x, Wg, Wu, Wd), atol=1e-8), "values differ: the gate goes through silu, the up projection does not"

check("swiglu_ffn", _test_swiglu_ffn, swiglu_ffn)

## Task 2: The ⅔ rule and real hidden sizes

Three matrices of size $d\times h$ cost $3dh$ parameters, while the ReLU FFN with $h = 4d$ costs $2\cdot d\cdot 4d = 8d^2$.
To keep the budget equal, set $h = \tfrac{2}{3}\cdot 4d$ (Shazeer: 3072 → 2048; LLaMA paper: "$\tfrac{2}{3}4d$ instead of $4d$").
Meta's code then optionally multiplies by `ffn_dim_multiplier` and rounds **up** to a multiple of `multiple_of`, which suits GPU tiles:
```
hidden = int(2 * (4 * d) / 3)
if ffn_dim_multiplier: hidden = int(ffn_dim_multiplier * hidden)
hidden = multiple_of * ceil(hidden / multiple_of)
```
Expected: LLaMA-7B (d = 4096, multiple_of = 256) gives **11,008**. Llama-3-8B (d = 4096, multiplier 1.3, multiple_of = 1024) gives **14,336**.

In [ ]:
def llama_hidden_dim(d, multiple_of=256, ffn_dim_multiplier=None):
    # TODO: follow the three lines above
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_llama_hidden_dim(f):
    assert f(4096) == 11008, f"LLaMA-7B: got {f(4096)}, expected 11008"
    assert f(5120) == 13824, f"LLaMA-13B: got {f(5120)}, expected 13824"
    assert f(8192) == 22016, f"LLaMA-65B: got {f(8192)}, expected 22016"
    assert f(4096, 1024, 1.3) == 14336, f"Llama-3-8B: got {f(4096, 1024, 1.3)}, expected 14336"
    assert f(768, 1) == 2048, f"Shazeer's T5-base setting (d=768, no rounding): got {f(768, 1)}, expected 2048"

check("llama_hidden_dim", _test_llama_hidden_dim, llama_hidden_dim)

## Task 3: Count the parameters

Return the number of weight parameters (no biases; modern LLMs drop them) of one FFN:
* `kind='relu'`: two matrices, $2dh$
* `kind='swiglu'`: three matrices, $3dh$

The test checks that SwiGLU at the ⅔ width costs the same as ReLU at $4d$ (up to rounding), and that the FFN is about two thirds of a Llama-3-8B layer.

In [ ]:
def ffn_params(d, h, kind):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_ffn_params(f):
    assert f(768, 3072, 'relu') == 768 * 3072 * 2
    assert f(768, 2048, 'swiglu') == f(768, 3072, 'relu'), "SwiGLU at 2048 must equal ReLU at 3072 for d = 768"
    ffn = f(4096, 14336, 'swiglu'); attn = 2 * 4096 * 4096 + 2 * 4096 * 1024
    assert ffn == 176_160_768, f"Llama-3-8B FFN params per layer: got {ffn:,}, expected 176,160,768"
    assert 0.80 < ffn / (ffn + attn) < 0.81, "the FFN should be about 81% of a Llama-3-8B layer's matmul parameters"

check("ffn_params", _test_ffn_params, ffn_params)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: The SwiGLU FFN
def silu_solution(z):
    return z / (1 + np.exp(-z))

def swiglu_ffn_solution(x, W_gate, W_up, W_down):
    return (silu_solution(x @ W_gate) * (x @ W_up)) @ W_down

check("swiglu_ffn (reference)", _test_swiglu_ffn, swiglu_ffn_solution)

In [ ]:
# @title Solution for Task 2: The ⅔ rule and real hidden sizes
def llama_hidden_dim_solution(d, multiple_of=256, ffn_dim_multiplier=None):
    hidden = int(2 * (4 * d) / 3)
    if ffn_dim_multiplier is not None:
        hidden = int(ffn_dim_multiplier * hidden)
    return multiple_of * ((hidden + multiple_of - 1) // multiple_of)

check("llama_hidden_dim (reference)", _test_llama_hidden_dim, llama_hidden_dim_solution)

In [ ]:
# @title Solution for Task 3: Count the parameters
def ffn_params_solution(d, h, kind):
    return {'relu': 2, 'swiglu': 3}[kind] * d * h

check("ffn_params (reference)", _test_ffn_params, ffn_params_solution)

---
## Experiments
**What does the gate do?** For a few random inputs, look at how many hidden units each FFN effectively switches off. ReLU zeroes units with a hard cut. SiLU is smooth, slightly negative for negative inputs, and the separate *up* projection can flip the sign of what passes through. Then compare Shazeer's reported held-out log-perplexities (T5-base on C4, lower is better).

In [ ]:
r = np.random.default_rng(0); d = 64; h_relu, h_glu = 256, llama_hidden_dim_solution(d, 1)
x = r.normal(size=(2000, d)); W1 = r.normal(0, d ** -.5, (d, h_relu)); Wg = r.normal(0, d ** -.5, (d, h_glu)); Wu = r.normal(0, d ** -.5, (d, h_glu))
relu_h = np.maximum(0, x @ W1); glu_h = silu_solution(x @ Wg) * (x @ Wu)
print(f"hidden units: ReLU {h_relu}, SwiGLU {h_glu} (same params: {2 * d * h_relu:,} vs {3 * d * h_glu:,})")
print(f"fraction of hidden activations exactly 0: ReLU {np.mean(relu_h == 0):.2f}, SwiGLU {np.mean(glu_h == 0):.2f}")
print(f"fraction negative:                        ReLU {np.mean(relu_h < 0):.2f}, SwiGLU {np.mean(glu_h < 0):.2f}")
print("\nShazeer (2020) Table 1, held-out log-perplexity:")
for name, a, b in [('ReLU (baseline)', 1.997, 1.677), ('GELU', 1.983, 1.679), ('GEGLU', 1.942, 1.633), ('SwiGLU', 1.944, 1.636)]:
    print(f"  {name:16} 65,536 steps: {a:.3f}   524,288 steps: {b:.3f}")

---
## Takeaways
* SwiGLU = SiLU-gated linear unit: a multiplicative gate plus a value projection, three matrices.
* The hidden size shrinks to about ⅔·4d so the parameter budget stays the same, then real models round up to hardware-friendly multiples.
* The FFN is about 80% of a Llama-3-8B layer's matmul parameters (the GQA attention is small). Most of a dense model's parameters and FLOPs live here, which is why Mixture-of-Experts (deck 10) replaces exactly this block.